# Simple RAG Evaluation Notebook

This notebook helps you evaluate the retrieval quality of your Retrieval-Augmented Generation (RAG) system for the Monster Resort Concierge project.

**Sections:**
1. Import required libraries
2. Load or define sample documents
3. Define retrieval function
4. Define evaluation question set
5. Retrieve relevant documents for each question
6. Evaluate retrieval results (precision, recall, F1)
7. Display evaluation metrics

**Instructions:**
- Run each cell in order.
- Manually score the relevance of each answer if ground truth is not available.
- Use this notebook to iteratively improve your RAG pipeline.

In [ ]:
# 1. Import Required Libraries
import os
import pandas as pd
from app.config import get_settings
from app.rag import VectorRAG


In [ ]:
# 2. Load or Define Sample Documents
settings = get_settings()
rag = VectorRAG(
    settings.rag_persist_dir,
    settings.rag_collection,
    embedding_model=getattr(settings, "embedding_model", "all-MiniLM-L6-v2")
)

# Optionally, print the number of docs in the collection
try:
    doc_count = rag.collection.count() if hasattr(rag.collection, 'count') else None
    print(f"Knowledge base contains {doc_count} documents.")
except Exception as e:
    print(f"Could not count documents: {e}")

In [ ]:
# 3. Define Retrieval Function

def retrieve(query, k=1):
    """Call the RAG search and return top-k results."""
    result = rag.search(query=query, k=k)
    if result.get("ok"):
        return result["results"]
    else:
        print(f"RAG search failed: {result}")
        return []

In [ ]:
# 4. Define Question Set for Evaluation
queries = [
    "What time is check-in?",
    "What amenities are available?",
    "Is breakfast included?",
    "What is the checkout policy?",
    "Is there parking at the resort?",
    "Can I bring pets?",
    "How do I request a late checkout?",
    "Are there accessible rooms?",
    "What is the cancellation policy?",
    "Is the pool open year-round?"
]

# Optionally, define ground truth answers or relevant doc snippets for each query (if available)
ground_truths = [None] * len(queries)  # Replace with actual answers if you have them

In [ ]:
# 5. Retrieve Relevant Documents for Each Question
results = []
for i, query in enumerate(queries):
    top_docs = retrieve(query, k=1)
    print(f"Q{i+1}: {query}")
    if top_docs:
        print(f"Top result: {top_docs[0]['text']}")
    else:
        print("No result found.")
    # Manual scoring: 0=irrelevant, 1=partially, 2=fully relevant
    score = int(input(f"Score relevance for Q{i+1} (0=irrelevant, 1=partial, 2=full): "))
    results.append({"query": query, "result": top_docs[0]["text"] if top_docs else None, "score": score})

In [ ]:
# 6. Evaluate Retrieval Results (Precision, Recall, F1)
import numpy as np
scores = [r["score"] for r in results]
avg_score = np.mean(scores)
print(f"\nAverage relevance score: {avg_score:.2f} (0=irrelevant, 2=fully relevant)")

# 7. Display Evaluation Metrics

The table below summarizes the manual evaluation of RAG retrieval quality for each query. Use this to identify strengths and weaknesses in your knowledge base or retrieval pipeline.

# 8. LLM-as-Judge: Automated Relevance Evaluation

This cell uses OpenAI to rate the relevance and faithfulness of each RAG answer. It helps you measure hallucination and answer quality automatically.

In [ ]:
# 8. LLM-as-Judge: Automated Relevance Evaluation
import openai
import time

def llm_judge(query, answer, model="gpt-4o"):
    prompt = f"""
You are an expert evaluator. Rate the following answer for relevance and faithfulness to the query. 

Query: {query}
Answer: {answer}

Respond with a JSON object: {{'relevance': 0-2, 'faithfulness': 0-2, 'explanation': '...'}}
- relevance: 0=irrelevant, 1=partial, 2=fully relevant
- faithfulness: 0=hallucinated, 1=partial, 2=fully faithful
"""
    try:
        response = openai.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a strict answer evaluator."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=128,
            temperature=0.0,
        )
        import json
        content = response.choices[0].message.content
        return json.loads(content)
    except Exception as e:
        print(f"LLM judge failed: {e}")
        return None

# Evaluate all results
auto_scores = []
for r in results:
    if r["result"]:
        score = llm_judge(r["query"], r["result"])
        auto_scores.append(score)
        print(f"Q: {r['query']}\nA: {r['result']}\nEval: {score}\n")
        time.sleep(1)  # avoid rate limits
    else:
        auto_scores.append(None)